<font color=white size=8 face=雅黑>**COHP分析的方法,
可以获取分轨道和绘图**</font>

In [1]:
from pathlib import Path
from pymatgen.electronic_structure.core import Spin
from VaspAnalysis import CohpAnalysis

# 1) 初始化 + 打印“加载信息/可用键列表”
work_dir = Path("/data2/home/luodh/cohp/cohp")
out_dir = Path("/data2/home/luodh/cohp")

cohp = CohpAnalysis(
    work_dir=work_dir,
    save_data=True,
    output_dir=out_dir
)

# 1) 检查：是否成功加载 COHPCAR / ICOHPLIST
print("COHPCAR loaded:", cohp.cohp_obj is not None)
print("ICOHPLIST loaded:", cohp.icohplist_obj is not None)

# 2) 可用 bond labels（非常关键：你要先知道有哪些键）
print("Available bond labels (first 20):", cohp.available_bond_labels[:20])
print("Total bond labels:", len(cohp.available_bond_labels))

# 2) 查询某个 bond 的所有分轨道标签（orbital interactions）
bond_label = cohp.available_bond_labels[0]
orb_list = cohp.get_orbital_bond_info(bond_label)
print(f"Bond {bond_label} has {len(orb_list)} orbital interactions.")
print("First 10 orbitals:", orb_list[:10])

# 3) 获取某个 (bond, orbital) 的原始数据块（包含 sites、COHP/ICOHP 等）
orbital_label = orb_list[0]  # 举例选第一个轨道相互作用标签
orb_data = cohp.get_orbital_data(bond_label, orbital_label)

print("orb_data keys:", list(orb_data.keys()))
print("sites:", orb_data.get("sites"))
print("has COHP:", "COHP" in orb_data)
print("has ICOHP:", "ICOHP" in orb_data)

# 查看自旋通道（COHP/ICOHP 内通常是 {Spin.up: array, Spin.down: array?}）
if "COHP" in orb_data:
    print("COHP channels:", list(orb_data["COHP"].keys()))


[Info] 加载 COHPCAR.lobster 成功：1 个 bond label。
[Info] 加载 ICOHPLIST.lobster 成功。
[Info] Bond '1' orbital interactions: ['6s-2s', '6s-2py', '6s-2pz', '6s-2px', '6py-2s', '6py-2py', '6py-2pz', '6py-2px', '6pz-2s', '6pz-2py', '6pz-2pz', '6pz-2px', '6px-2s', '6px-2py', '6px-2pz', '6px-2px', '5dxy-2s', '5dxy-2py', '5dxy-2pz', '5dxy-2px', '5dyz-2s', '5dyz-2py', '5dyz-2pz', '5dyz-2px', '5dz2-2s', '5dz2-2py', '5dz2-2pz', '5dz2-2px', '5dxz-2s', '5dxz-2py', '5dxz-2pz', '5dxz-2px', '5dx2-2s', '5dx2-2py', '5dx2-2pz', '5dx2-2px']


COHPCAR loaded: True
ICOHPLIST loaded: True
Available bond labels (first 20): ['1']
Total bond labels: 1
Bond 1 has 36 orbital interactions.
First 10 orbitals: ['6s-2s', '6s-2py', '6s-2pz', '6s-2px', '6py-2s', '6py-2py', '6py-2pz', '6py-2px', '6pz-2s', '6pz-2py']
orb_data keys: ['COHP', 'ICOHP', 'orbitals', 'length', 'sites', 'cells']
sites: (17, 64)
has COHP: True
has ICOHP: True
COHP channels: [<Spin.up: 1>]


<font color=white size=8 face=雅黑>**导出轨道信息为CSV文件/图片**</font>

In [ ]:
# 4) 导出 CSV：分轨道 COHP/ICOHP（单轨道/多轨道统一）+ 总键 COHP/ICOHP
# 4.1 分轨道：导出单轨道 COHP（Spin.up）——仍然只需传 orbitals=单个字符串
cohp.save_orbital_data_to_csv(
    bond_label=bond_label,
    orbitals=orbital_label,   # 单轨道：str
    data_type="COHP",
    spin=Spin.up,
    output_filename=None,     # None => 自动命名
)

# 4.2 分轨道：导出单轨道 ICOHP（Spin.up）
cohp.save_orbital_data_to_csv(
    bond_label=bond_label,
    orbitals=orbital_label,   # 单轨道：str
    data_type="ICOHP",
    spin=Spin.up,
)

# 4.3 总键：导出 Total COHP（Spin.up）
cohp.save_total_bond_data_to_csv(
    bond_label=bond_label,
    data_type="COHP",
    spin=Spin.up,
)

# 4.4 分轨道：导出多轨道到同一个 CSV（多列）——PPT/数据汇总很方便
multi_orbitals = orb_list[:6] if len(orb_list) >= 6 else orb_list
print("Multi orbitals for CSV:", multi_orbitals)

cohp.save_orbital_data_to_csv(
    bond_label=bond_label,
    orbitals=multi_orbitals,     # 多轨道：list[str]
    data_type="COHP",
    spin="both",                 # 自旋非极化时会自动退化为 up
    minus_cohp=True,             # 导出 -COHP（常用）
    output_filename=out_dir / f"multi_orbitals_minusCOHP_bond{bond_label}.csv",
)

[Saved] Bond1_selected_COHP_up.csv
[Saved] Bond1_selected_ICOHP_up.csv


[Saved] TotalBond1_COHP_up.csv
[Saved] /data2/home/luodh/cohp/multi_orbitals_minusCOHP_bond1.csv


Multi orbitals for CSV: ['6s-2s', '6s-2py', '6s-2pz', '6s-2px', '6py-2s', '6py-2py']


,Energy (E-Ef),6s-2s_up,6s-2py_up,6s-2pz_up,6s-2px_up,6py-2s_up,6py-2py_up
0,-20.02503,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
1,-19.97497,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
2,-19.92491,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
3,-19.87484,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
4,-19.82478,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
...,...,...,...,...,...,...,...
796,19.82478,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
797,19.87484,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
798,19.92491,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
799,19.97497,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0


<font color=white size=8 face=雅黑>**导出轨道信息为高质量图片**</font>

In [3]:
# 5) 高质量绘图：分轨道 & 总键
# 5.1 分轨道 COHP：单轨道绘图（统一用 plot_cohp）
fig, ax = cohp.plot_cohp(
    bond_label=bond_label,
    orbitals=orbital_label,  # 单轨道也用同一个函数
    data_type="COHP",
    spin="both",
    xlim=(-10, 5),
    minus_cohp=True,
    fill=True,
    beta_dashed=True,
    savepath=out_dir / f"orbital_COHP_bond{bond_label}.pdf",
    show=False,
)

# 5.1(补充) 同一张图保存 PNG（PPT 更方便）
fig, ax = cohp.plot_cohp(
    bond_label=bond_label,
    orbitals=orbital_label,
    data_type="COHP",
    spin="both",
    xlim=(-10, 5),
    minus_cohp=True,
    fill=True,
    beta_dashed=True,
    savepath=out_dir / f"orbital_COHP_bond{bond_label}.png",
    dpi=600,
    show=False,
)

# 5.2 总键 COHP：同图绘 up/down（仍用总键函数）
fig, ax = cohp.plot_total_bond_cohp(
    bond_label=bond_label,
    data_type="COHP",
    spin="both",
    xlim=(-10, 5),
    minus_cohp=True,
    fill=True,
    savepath=out_dir / f"total_COHP_bond{bond_label}.png",
    dpi=600,
    show=False,
)

# 5.3 分轨道 ICOHP：单轨道同图绘 up/down（ICOHP 单位 eV）
fig, ax = cohp.plot_cohp(
    bond_label=bond_label,
    orbitals=orbital_label,
    data_type="ICOHP",
    spin="both",
    xlim=(-10, 5),
    fill=True,
    savepath=out_dir / f"orbital_ICOHP_bond{bond_label}.png",
    dpi=600,
    show=False,
)

# 5.4 分轨道 COHP：多轨道同图（多曲线）——仍然同一个 plot_cohp
fig, ax = cohp.plot_cohp(
    bond_label=bond_label,
    orbitals=multi_orbitals,
    data_type="COHP",
    spin="both",
    xlim=(-10, 5),
    minus_cohp=True,
    mirror_down=False,   # COHP 通常不必镜像；想要 DOS 风格可 True
    fill=False,          # 多曲线建议 False（更清爽）
    savepath=out_dir / f"multi_orbitals_COHP_bond{bond_label}.png",
    dpi=600,
    show=False,
)

# 6) 读取 ICOHPLIST 的摘要信息（平均 ICOHP、键长、键数量）
summary = cohp.get_icohp_summary(bond_label=bond_label)
print("ICOHP summary:", summary)

# 7) 高通量批处理示例：批量画多个 bond 的总 COHP（保存 PNG）
for bl in cohp.available_bond_labels[:10]:
    cohp.plot_total_bond_cohp(
        bond_label=bl,
        data_type="COHP",
        spin="both",
        xlim=(-10, 5),
        minus_cohp=True,
        fill=True,
        savepath=out_dir / f"total_COHP_bond{bl}.png",
        dpi=450,
        show=False,
    )

print("All done. Outputs saved to:", out_dir)

[Plot] Saved to /data2/home/luodh/cohp/orbital_COHP_bond1.pdf
[Plot] Saved to /data2/home/luodh/cohp/orbital_COHP_bond1.png
[Plot] Saved to /data2/home/luodh/cohp/total_COHP_bond1.png
[Plot] Saved to /data2/home/luodh/cohp/orbital_ICOHP_bond1.png
[Plot] Saved to /data2/home/luodh/cohp/multi_orbitals_COHP_bond1.png


ICOHP summary: {'label': '1', 'icohp_average': {<Spin.up: 1>: -4.27981}, 'length': 2.07801, 'number_of_bonds': 1, 'is_spin_polarized': False}


[Plot] Saved to /data2/home/luodh/cohp/total_COHP_bond1.png


All done. Outputs saved to: /data2/home/luodh/cohp
